# Univariate Distances — All 100 Datasets (Chap)

For each of the 100 synthetic datasets (50 ECDF + 50 CTGAN),  
compute three distance metrics per numerical variable between the **training set** and the **synthetic set**:

| Metric | Description |
|---|---|
| Cosine | Cosine distance between variable vectors |
| Jensen-Shannon | JSD between fitted normal PDFs |
| Wasserstein | Earth-mover distance between empirical distributions |

All three metrics are normalised independently to [0, 1] for visualisation.

In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
import matplotlib.pyplot as plt

# ── paths ──────────────────────────────────────────────────────────────────
HOME_PATH      = '../../../functions/evaluation_functions/'
FUNCTIONS_DIR  = 'EVALUATION FUNCTIONS/RESEMBLANCE'
META_ECDF      = '../../../data/processed/chap/ensemble_50run/metadata/run_log_ecdf.csv'
META_CTGAN     = '../../../data/processed/chap/ensemble_50run/metadata/run_log_ctgan.csv'
OUT_DIR        = os.path.join(os.getcwd(), 'UNIVARIATE RESEMBLANCE RESULTS')
os.makedirs(OUT_DIR, exist_ok=True)

# ── import evaluation functions ────────────────────────────────────────────
ACTUAL_DIR = os.getcwd()
os.chdir(HOME_PATH + FUNCTIONS_DIR)
from univariate_resemblance import scale_data, cosine_distances, js_distances, wass_distances
os.chdir(ACTUAL_DIR)

## 1. Build dataset pairs from metadata

In [3]:
log_ecdf  = pd.read_csv(META_ECDF)
log_ctgan = pd.read_csv(META_CTGAN)

# training set path for each imputation id (from ecdf log)
train_map = (
    log_ecdf[log_ecdf['stage'] == 'train']
    .set_index('imp_id')['file_path']
    .to_dict()
)

# ── ECDF pairs ──────────────────────────────────────────────────────────────
ecdf_syn = log_ecdf[log_ecdf['stage'] == 'ecdf_syn'].copy()
ecdf_pairs = ecdf_syn[['imp_id', 'syn_id', 'file_path']].rename(columns={'file_path': 'syn_path'})
ecdf_pairs = ecdf_pairs.assign(method='ecdf', train_path=ecdf_pairs['imp_id'].map(train_map))

# ── CTGAN pairs ─────────────────────────────────────────────────────────────
ctgan_syn = log_ctgan[log_ctgan['stage'] == 'ctgan_syn'].copy()
ctgan_pairs = ctgan_syn[['imp_id', 'syn_id', 'file_path']].rename(columns={'file_path': 'syn_path'})
ctgan_pairs = ctgan_pairs.assign(method='ctgan', train_path=ctgan_pairs['imp_id'].map(train_map))

# ── combine ─────────────────────────────────────────────────────────────────
all_pairs = pd.concat([ecdf_pairs, ctgan_pairs], ignore_index=True)
all_pairs = all_pairs[['method', 'imp_id', 'syn_id', 'train_path', 'syn_path']]

print(f"Total dataset pairs: {len(all_pairs)}  "
      f"(ECDF: {(all_pairs.method=='ecdf').sum()}, CTGAN: {(all_pairs.method=='ctgan').sum()})")
all_pairs

Total dataset pairs: 100  (ECDF: 50, CTGAN: 50)


,method,imp_id,syn_id,train_path,syn_path
0,ecdf,1.0,1.0,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...
1,ecdf,1.0,2.0,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...
2,ecdf,1.0,3.0,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...
3,ecdf,1.0,4.0,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...
4,ecdf,1.0,5.0,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...
...,...,...,...,...,...
95,ctgan,5.0,6.0,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...
96,ctgan,5.0,7.0,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...
97,ctgan,5.0,8.0,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...
98,ctgan,5.0,9.0,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...,/Users/hazel/Dropbox/jinyuan/synthetic_data/re...


## 2. Compute distances for every dataset × variable

For each pair the data is scaled independently to [0, 1] per variable  
(consistent with the single-dataset notebook), then three distances are computed.

In [10]:
results = []

for idx, row in all_pairs.iterrows():
    # load datasets
    train = pd.read_csv(row['train_path'])
    syn   = pd.read_csv(row['syn_path'])

    # numerical columns only
    num_cols = train.select_dtypes(include=['float64', 'int64']).columns.tolist()

    # scale each dataset to [0,1] per variable
    train_scaled = scale_data(train[num_cols])
    syn_scaled   = scale_data(syn[num_cols])

    # compute distances (each returns a list of len = n_numerical_vars)
    cos  = cosine_distances(train_scaled, syn_scaled)
    js   = js_distances(train_scaled, syn_scaled)
    wass = wass_distances(train_scaled, syn_scaled)

    for i, col in enumerate(num_cols):
        results.append({
            'method':       row['method'],
            'imp_id':       row['imp_id'],
            'syn_id':       row['syn_id'],
            'variable':     col,
            'cosine':       cos[i],
            'js':           js[i],
            'wasserstein':  wass[i],
        })

    if (idx + 1) % 10 == 0:
        print(f"  processed {idx+1}/{len(all_pairs)} pairs")

df_results = pd.DataFrame(results)
print(f"\nResult shape: {df_results.shape}")
df_results

  processed 10/100 pairs
  processed 20/100 pairs
  processed 30/100 pairs
  processed 40/100 pairs
  processed 50/100 pairs
  processed 60/100 pairs
  processed 70/100 pairs
  processed 80/100 pairs
  processed 90/100 pairs
  processed 100/100 pairs

Result shape: (2000, 7)


,method,imp_id,syn_id,variable,cosine,js,wasserstein
0,ecdf,1.0,1.0,sbp_bin1,0.260550,0.039274,0.005987
1,ecdf,1.0,1.0,sbp_bin2,0.160776,0.035440,0.004358
2,ecdf,1.0,1.0,sbp_bin3,0.117736,0.033674,0.049116
3,ecdf,1.0,1.0,sbp_bin4,0.151418,0.037514,0.008612
4,ecdf,1.0,1.0,sbp_bin5,0.099262,0.030050,0.004020
...,...,...,...,...,...,...,...
1995,ctgan,5.0,10.0,dbp_bin6,0.095569,0.038699,0.086171
1996,ctgan,5.0,10.0,dbp_bin7,0.081702,0.032162,0.008732
1997,ctgan,5.0,10.0,dbp_bin8,0.069415,0.028074,0.018529
1998,ctgan,5.0,10.0,dbp_bin9,0.088759,0.031031,0.056808


## 3. Save raw results

In [5]:
out_raw = os.path.join(OUT_DIR, 'distances_all_datasets_chap.csv')
df_results.to_csv(out_raw, index=False)
print(f"Saved: {out_raw}")
df_results.describe()

Saved: /Users/hazel/Dropbox/jinyuan/synthetic_data/review_paper/STDG-evaluation-metrics/notebooks/Dataset 1 - Chap/Synthetic data evaluation/Resemblance/UNIVARIATE RESEMBLANCE RESULTS/distances_all_datasets_chap.csv


,imp_id,syn_id,cosine,js,wasserstein
count,2000.000000,2000.000,2000.000000,2000.000000,2000.000000
mean,3.000000,5.500,0.109621,0.034099,0.047999
std,1.414567,2.873,0.045869,0.008243,0.055195
min,1.000000,1.000,0.016167,0.015238,0.000962
25%,2.000000,3.000,0.081780,0.029312,0.006068
50%,3.000000,5.500,0.101688,0.034294,0.035076
75%,4.000000,8.000,0.121480,0.038209,0.062701
max,5.000000,10.000,0.299496,0.063561,0.302446
